# 06 — Body Weight Model Robustness and Calibration

This evaluation asks whether the 7-day V1 result is stable enough to revisit—not whether synthetic data can be tuned until ML wins. V1 previously found Recent Trend MAE ≈0.334 kg, Last Value ≈0.431 kg, and Linear Regression ≈0.604 kg on the deterministic synthetic fixture. The horizon remains 7 days (±2-day target tolerance) with five prior measurements.

The analysis adds Ridge, temporal periods and cutoffs, cadence/history segments, causal empirical intervals, explicit abstention, error analysis, and fixed sensitivity checks. Synthetic results validate mechanics only.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from metaform_ds.config import DataScienceConfig
from metaform_ds.modeling.weight import WeightModelConfig, evaluate_weight_robustness, render_robustness_report
from metaform_ds.modeling.weight.targets import build_weight_prediction_examples
from metaform_ds.utils.notebooks import load_notebook_data
settings = DataScienceConfig.from_env(); data = load_notebook_data(settings)
origin = 'SYNTHETIC' if data.mode == 'SYNTHETIC' else 'REAL'
print(f'DATA ORIGIN: {origin}')
if origin == 'SYNTHETIC': print('SYNTHETIC ROBUSTNESS VALIDATION — not real-world performance evidence.')
else: print('REAL-WORLD ROBUSTNESS EVALUATION — read-only pseudonymized data.')

In [ ]:
config = WeightModelConfig(); target_result = build_weight_prediction_examples(data.body_weight, config)
dates = target_result.examples['prediction_date'].sort_values() if not target_result.examples.empty else pd.Series(dtype='datetime64[ns, UTC]')
cutoffs = [pd.Timestamp(value) for value in dates.quantile([0.33, 0.50, 0.67]).drop_duplicates()] if len(dates) else []
result = evaluate_weight_robustness(data.body_weight, data_origin=origin, config=config, calendar_cutoffs=cutoffs)
print(f'Status: {result.status}; predictions: {len(result.predictions)}; preferred: {result.preferred_method}')
print(f'Recommendation state: {result.recommendation_state}')
result.comparison

## Temporal robustness and model comparison

In [ ]:
print('Fold stability:'); print(result.stability.to_string(index=False))
print('\nEarly/middle/late periods:'); print(result.temporal_periods.to_string(index=False))
print('\nCalendar cutoffs:'); print(result.calendar_cutoffs.to_string(index=False))

In [ ]:
if result.fold_metrics.empty: print('Insufficient fold metrics.')
else:
    fig, ax = plt.subplots()
    for model, group in result.fold_metrics.groupby('model'): ax.plot(group['prediction_date'], group['mae_kg'], marker='o', label=model)
    ax.set(title='Fold MAE by temporal origin', xlabel='Prediction date (UTC)', ylabel='Fold MAE (kg)'); ax.legend(); fig.autofmt_xdate(); plt.show()

In [ ]:
if not result.comparison.empty:
    view = result.comparison.sort_values('mae_kg')
    fig, ax = plt.subplots(); ax.bar(view['model'], view['mae_kg'])
    ax.set(title='Aggregate temporal MAE', xlabel='Method', ylabel='MAE (kg)'); ax.tick_params(axis='x', rotation=20); plt.show()

## Cadence and available-history analysis

In [ ]:
print(result.cadence_metrics.to_string(index=False))
if not result.cadence_metrics.empty:
    view = result.cadence_metrics.pivot(index='cadence_bucket', columns='model', values='mae_kg')
    fig, ax = plt.subplots(); view.plot.bar(ax=ax)
    ax.set(title='MAE by past-only measurement cadence', xlabel='Cadence bucket', ylabel='MAE (kg)'); ax.tick_params(axis='x', rotation=0); plt.show()

In [ ]:
print(result.history_metrics.to_string(index=False))
if not result.history_metrics.empty:
    view = result.history_metrics.pivot(index='history_bucket', columns='model', values='mae_kg')
    fig, ax = plt.subplots(); view.plot.bar(ax=ax)
    ax.set(title='MAE by prior-history length', xlabel='Prior observations', ylabel='MAE (kg)'); ax.tick_params(axis='x', rotation=0); plt.show()

## Empirical uncertainty and abstention
Intervals use only earlier out-of-fold residuals whose target was already observed before each prediction time. Nominal 80% coverage is not guaranteed observed coverage.

In [ ]:
print(result.interval_diagnostics.to_string(index=False))
preferred_intervals = result.intervals[result.intervals['model'].eq(result.preferred_method)]
available = preferred_intervals[preferred_intervals['interval_available']]
if available.empty: print('Insufficient prior residuals: no causal empirical intervals are available.')
else:
    fig, ax = plt.subplots(); ax.plot(available['target_date'], available['actual_weight_kg'], marker='o', label='Actual'); ax.plot(available['target_date'], available['point_prediction_kg'], marker='o', label='Prediction'); ax.fill_between(available['target_date'], available['lower_bound_kg'], available['upper_bound_kg'], alpha=0.2, label='Empirical interval')
    ax.set(title='Actual and predicted weight with empirical interval', xlabel='Target date (UTC)', ylabel='Weight (kg)'); ax.legend(); fig.autofmt_xdate(); plt.show()

In [ ]:
if not available.empty:
    widths = available['upper_bound_kg'] - available['lower_bound_kg']
    fig, ax = plt.subplots(); ax.hist(widths, bins=min(10, max(3, len(widths))))
    ax.set(title='Empirical interval width distribution', xlabel='Interval width (kg)', ylabel='Predictions'); plt.show()
print('Reliability metadata:'); print(result.reliability_metadata.drop(columns=['anonymous_user_id']).to_string(index=False))

## Error and sensitivity analysis

In [ ]:
largest = result.errors.head(10); print('Largest retained errors:'); print(largest.drop(columns=['anonymous_user_id']).to_string(index=False))
preferred_errors = result.errors[result.errors['model'].eq(result.preferred_method)]
if not preferred_errors.empty:
    fig, ax = plt.subplots(); ax.plot(preferred_errors['prediction_date'], preferred_errors['absolute_error_kg'], marker='o')
    ax.set(title=f'Absolute error over time — {result.preferred_method}', xlabel='Prediction date (UTC)', ylabel='Absolute error (kg)'); fig.autofmt_xdate(); plt.show()

In [ ]:
print(result.sensitivity.to_string(index=False))
sensitivity_view = result.sensitivity[result.sensitivity['model'].eq('Recent Trend Baseline')]
if not sensitivity_view.empty:
    fig, ax = plt.subplots(); ax.scatter(sensitivity_view['evaluated_predictions'], sensitivity_view['mae_kg'])
    for _, row in sensitivity_view.iterrows(): ax.annotate(f"h={int(row['minimum_prior_measurements'])}, tol={int(row['target_tolerance_days'])}", (row['evaluated_predictions'], row['mae_kg']))
    ax.set(title='Sensitivity tradeoff for Recent Trend', xlabel='Evaluated predictions', ylabel='MAE (kg)'); plt.show()

## Production recommendation and limitations

In [ ]:
print(f'Preferred method: {result.preferred_method}')
print(f'Recommendation state: {result.recommendation_state}')
print('A more complex model is not automatically better. Chronology, uncertainty availability, fold stability, sample size, and baseline gains all remain gates.')
if origin == 'SYNTHETIC': print('Real-world robustness evaluation pending. No application integration is justified.')
report = render_robustness_report(result)
artifact_dir = settings.artifact_dir
if not artifact_dir.is_absolute() and Path.cwd().name == 'notebooks': artifact_dir = Path('..') / artifact_dir
artifact_dir.mkdir(parents=True, exist_ok=True)
report_path = artifact_dir / 'body_weight_v1_robustness.md'; report_path.write_text(report, encoding='utf-8')
print(f'Ignored aggregate report written to: {report_path}')
print(report)